# Syrinx — Exploratory Analysis

Interactive notebook for exploring pipeline outputs.
Run `python run_pipeline.py` first to populate `data/` and `outputs/`.

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from syrinx.config import load_config
from syrinx.descriptive import compute_entropy_measures
from syrinx.utils import load_array

cfg = load_config('config.yaml')
print(f'Config hash: {cfg.hash}')

## Download manifest

In [ ]:
manifest_path = Path(cfg.data_dir) / 'manifests' / 'download_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    records = manifest.get('records', [])
    df = pd.DataFrame(records)
    print(f'{len(df)} recordings downloaded')
    if not df.empty:
        display(df[['xc_id', 'species', 'country', 'lat', 'lon', 'licence_type', 'grade_b']].head(20))
else:
    print('No download manifest found — run acquire stage first')

## Acoustic distance matrix

In [ ]:
dist_path = Path(cfg.data_dir) / 'distances' / 'acoustic_distance_genus.pkl'
if dist_path.exists():
    data = load_array(dist_path)
    D = data['D']
    entities = data['entities']
    fig = px.imshow(
        D,
        labels={'x': 'Species', 'y': 'Species', 'color': 'Acoustic distance'},
        x=entities, y=entities,
        title='Pairwise acoustic distance matrix',
        color_continuous_scale='Viridis_r',
    )
    fig.show()
else:
    print('Distance matrix not found — run alignment stage first')

## Inference results

In [ ]:
for hyp in ['h1', 'h2', 'h3']:
    path = Path(cfg.output_dir) / f'results_{hyp}.json'
    if path.exists():
        res = json.loads(path.read_text())
        print(f'\n=== {hyp.upper()} ===')
        interp = res.get('interpretation', res.get('status', 'no interpretation'))
        print(f'  Interpretation: {interp}')
        if 'mrm' in res:
            mrm = res['mrm']
            print(f'  MRM coefficient: {mrm.get("molecular_coefficient")}')
            print(f'  MRM p-value:     {mrm.get("p_value")}')
            print(f'  Semi-partial r²: {mrm.get("semipartial_r2")}')
        if 'rho' in res:
            print(f'  Spearman ρ:      {res.get("rho")}')
            print(f'  p-value:         {res.get("p_value")}')
        if 'mantel_pearson' in res:
            print(f'  Mantel r:        {res["mantel_pearson"].get("r")}')
            print(f'  Mantel p:        {res["mantel_pearson"].get("p_value")}')
    else:
        print(f'{hyp.upper()}: not run yet')

## Species profiles

In [ ]:
profile_path = Path(cfg.output_dir) / 'species_profile.json'
if profile_path.exists():
    profiles = json.loads(profile_path.read_text())
    rows = []
    for sp, p in profiles.items():
        rows.append({
            'species': sp,
            'H1': p['entropy'].get('H1'),
            'H2': p['entropy'].get('H2'),
            'C': p['entropy'].get('complexity_C'),
            'peak_freq_hz': p['acoustic'].get('mean_peak_freq_hz'),
        })
    df = pd.DataFrame(rows).sort_values('C', ascending=False)
    display(df.head(20))
    fig = px.scatter(df, x='H1', y='H2', color='C',
                     hover_data=['species'],
                     title='Song complexity: H1 vs H2',
                     labels={'H1': 'H₁ (unigram entropy)', 'H2': 'H₂ (conditional entropy)'})
    fig.show()
else:
    print('Species profiles not found — run full pipeline first')